# <font color="steelblue">22. Teorema de Bayes: ¿cuánto acierta un detector de bulos?</font>

*Unidad 4. Probabilidad* · Sesión teórica 22 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (ninguno) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = []
PREGUNTAS = ["q_P22_1.json", "q_P22_2.json", "q_P22_3.json", "q_P22_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

Una empresa anuncia un **detector automático de bulos con un 95 % de acierto**. ¿Si marca una noticia como falsa, hay un 95 % de probabilidad de que lo sea? Casi nunca: depende de **cuántos bulos hay** entre lo que se analiza. El **teorema de Bayes** da la respuesta correcta y explica también por qué fallan los cribados masivos o los sistemas de reconocimiento facial.

**Qué necesitas saber antes**: el cuaderno 21.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular una probabilidad total con un árbol.
2. Aplicar el teorema de Bayes para invertir una probabilidad condicionada.
3. Entender sensibilidad, especificidad y valor predictivo de un sistema de detección.
4. Explicar el papel de la prevalencia (la frecuencia de lo que se busca).

## <font color='steelblue'>Contenidos</font>

1. Sensibilidad, especificidad y prevalencia
2. Probabilidad total con un árbol
3. El teorema de Bayes
4. La prevalencia lo cambia todo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Sensibilidad, especificidad y prevalencia</font>

Cualquier sistema que clasifica en «sí/no» (un detector de bulos, una prueba médica, un filtro de spam) tiene dos tipos de acierto:

| Medida | Qué es | Ejemplo |
| :-- | :-- | :-- |
| **Sensibilidad** | P(marca bulo \| es bulo) | detecta el 95 % de los bulos |
| **Especificidad** | P(marca verdadero \| es verdadero) | deja pasar el 95 % de lo verdadero |
| **Prevalencia** | P(es bulo) | el 2 % de lo que se analiza son bulos |

Lo que le interesa al periodista es otra cosa: el **valor predictivo positivo**, P(es bulo \| marca bulo).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P22_1.json")

## <font color='steelblue'>2. Probabilidad total con un árbol</font>

¿Qué proporción de noticias marcará el detector como bulo? Hay dos caminos: ser bulo y ser detectado, o ser verdadera y ser marcada por error.

$$P(\text{marca}) = P(\text{bulo})\,P(\text{marca} \mid \text{bulo}) + P(\text{verdad})\,P(\text{marca} \mid \text{verdad})$$

In [ ]:
prevalencia = 0.02
sensibilidad = 0.95
especificidad = 0.95

p_marca = prevalencia * sensibilidad + (1 - prevalencia) * (1 - especificidad)
print(f"P(el detector marca bulo) = {p_marca:.4f}")

**Qué ha pasado**: aunque solo el 2 % son bulos, el detector marca cerca del **7 %** de las noticias, porque el 5 % de error sobre el 98 % de noticias verdaderas es una cantidad grande.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P22_2.json")

## <font color='steelblue'>3. El teorema de Bayes</font>

El **teorema de Bayes** invierte la condicionada:

$$P(\text{bulo} \mid \text{marca}) = \frac{P(\text{bulo})\, P(\text{marca} \mid \text{bulo})}{P(\text{marca})}$$

In [ ]:
vpp = prevalencia * sensibilidad / p_marca
print(f"P(es bulo | el detector lo marca) = {vpp:.3f}")

**Qué ha pasado**: ¡solo el 28 % de lo marcado es realmente un bulo! Con una prevalencia baja, la mayoría de las alarmas son **falsos positivos**. Un titular como «el algoritmo detecta bulos con un 95 % de fiabilidad» es, por tanto, engañoso si no dice cuántos bulos hay en lo que analiza.

Lo comprobamos con frecuencias naturales, que se entienden mejor que las fórmulas. Imagina 10.000 noticias:

In [ ]:
n = 10_000
bulos = n * prevalencia
verdaderas = n - bulos
bulos_detectados = bulos * sensibilidad
falsas_alarmas = verdaderas * (1 - especificidad)
print(f"De {n} noticias: {bulos:.0f} bulos y {verdaderas:.0f} verdaderas")
print(f"Marcadas como bulo: {bulos_detectados:.0f} bulos de verdad + {falsas_alarmas:.0f} falsas alarmas")
print(f"Proporción de marcadas que son bulo: {bulos_detectados / (bulos_detectados + falsas_alarmas):.3f}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P22_3.json")

## <font color='steelblue'>4. La prevalencia lo cambia todo</font>

**Qué vamos a hacer**: calcular el valor predictivo positivo para distintas prevalencias, con el mismo detector.

In [ ]:
def valor_predictivo(prev, sens=0.95, esp=0.95):
    return prev * sens / (prev * sens + (1 - prev) * (1 - esp))

prevs = np.linspace(0.005, 0.6, 100)
plt.plot(prevs * 100, [valor_predictivo(p) * 100 for p in prevs], color="steelblue")
plt.xlabel("Prevalencia: % de bulos en lo analizado")
plt.ylabel("% de alarmas que son bulos de verdad")
plt.title("El mismo detector (95 % / 95 %) según cuántos bulos haya")
plt.grid(alpha=0.3)
plt.show()

for p in [0.01, 0.1, 0.5]:
    print(f"Prevalencia {p:.0%}: valor predictivo {valor_predictivo(p):.1%}")

Si el detector se aplica solo a contenidos ya sospechosos (alta prevalencia), sus alarmas son mucho más fiables. Lo mismo pasa con las pruebas médicas: no es igual aplicarlas a toda la población que a personas con síntomas.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P22_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 3)*

Un sistema de reconocimiento facial tiene un 99 % de sensibilidad y un 99 % de especificidad. Se aplica a 100.000 personas en un estadio, entre las que hay 10 buscadas. ¿Qué proporción de las alarmas corresponde a personas buscadas?

<details><summary>Ver solución</summary>

```python
valor_predictivo(10 / 100_000, sens=0.99, esp=0.99)
```

Menos del 1 %: casi todas las alarmas son falsas.
</details>

### **Ejercicio 2** *(sección 3)*

Repite el cálculo con frecuencias naturales (cuántas alarmas verdaderas y cuántas falsas).

<details><summary>Ver solución</summary>

```python
print(10 * 0.99, (100_000 - 10) * 0.01)
```
</details>

### **Ejercicio 3** *(sección 4)*

¿Qué especificidad necesitaría el detector de bulos (con sensibilidad 0,95 y prevalencia 0,02) para que el 80 % de sus alarmas fueran bulos reales? Prueba valores con un bucle.

<details><summary>Ver solución</summary>

```python
for esp in [0.95, 0.98, 0.99, 0.995, 0.999]:
    print(esp, round(valor_predictivo(0.02, 0.95, esp), 3))
```
</details>

### **Ejercicio 4** *(sección 1)*

Escribe un párrafo para una noticia que explique, sin fórmulas, por qué «95 % de acierto» no significa que el 95 % de las alarmas sean ciertas.

<details><summary>Ver idea clave</summary>

Usa frecuencias naturales: «de cada 10.000 noticias...».
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **Sensibilidad** = P(detecta | es); **especificidad** = P(no detecta | no es); **prevalencia** = P(es).
- **Probabilidad total**: se suman los caminos del árbol que llevan a un resultado.
- **Bayes** invierte la condicionada: P(es | detecta) = P(es) · P(detecta | es) / P(detecta).
- Con **prevalencia baja**, hasta un buen detector da mayoría de **falsos positivos**.
- Explica estos resultados con **frecuencias naturales** («de cada 10.000...»).

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Sensibilidad** | Proporción de positivos reales que el sistema detecta. |
| **Especificidad** | Proporción de negativos reales que el sistema reconoce como tales. |
| **Prevalencia** | Proporción de casos positivos en la población analizada. |
| **Valor predictivo positivo** | Probabilidad de que un caso marcado sea realmente positivo. |
| **Falso positivo** | Caso marcado como positivo que en realidad no lo es. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 23 verás las distribuciones binomial y de Poisson.

**Siguiente cuaderno**: [23. Variables aleatorias discretas: binomial y Poisson](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P23_binomial_y_poisson.ipynb)